# Notebook 07: Expanded matched model comparison
This notebook widens the matched comparison from Notebook 04. It adds five classical models (logistic regression, random forest, histogram gradient boosting, k-nearest neighbours and a small MLP) and one quantum-kernel variant (a Z feature map without entanglement) next to the existing dummy, linear/RBF/polynomial SVMs and the ZZ-map QSVM. A projected quantum kernel can be switched on as an optional seventh model.

Every model uses the extension protocol unchanged: the same five stocks, the same 1,000 training, 400 validation and 500 evaluation rows, the same 2- or 4-component angle-scaled PCA inputs fitted on training rows only, hyperparameters chosen only by validation balanced accuracy, and a refit on the 1,400 development rows. The evaluation period was already inspected in earlier notebooks, so all results here are exploratory. Shared code is in `experiments/expanded_comparison.py`; Notebooks 01-06 and `reports/stage5` are not changed.

Unlike Notebook 06, which trains classical models on the full five-stock panel and the QSVM on a smaller date sample, every model here sees exactly the same rows. That makes this the row-matched comparison.

### Cell 1: Configure and load the shared code

Set the project root, choose whether to include the projected quantum kernel, and import the experiment module.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'experiments'))
import expanded_comparison as ec
PROJECTED = False  # set True to add the projected quantum kernel as a seventh new model
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print('Models:', list(ec.grids(PROJECTED)))

## Models and fairness
| Model | Family | Grid (validation-selected) | Notes |
|---|---|---|---|
| Logistic regression | classical | C in {0.01, 0.1, 1, 10} | balanced class weights |
| Random forest | classical | max_depth in {3, 5, None} x min_samples_leaf in {1, 10}, 300 trees | balanced class weights; seed check over 5 extra seeds |
| Gradient boosting | classical | learning rate {0.05, 0.1} x depth {2, 3} x iterations {100, 200} | balanced class weights, no early stopping |
| k-nearest neighbours | classical | k in {5, 15, 31} x uniform/distance weights | no class-weight option in scikit-learn |
| MLP | classical | one hidden layer of 8 or 16 units x alpha {0.001, 0.1} | no class-weight option; seed check over 5 extra seeds |
| QSVM, Z map | quantum | C in {0.1, 1, 10} | exact statevector, no entanglement; isolates the ZZ entangling layer |
| Projected kernel (optional) | quantum | C {0.1, 1, 10} x gamma {0.25, 1, 4} | RBF on single-qubit Bloch vectors of the ZZ-map state |

The grids are small and declared before any evaluation run. They match observations and representations, not tuning compute: tree ensembles and the MLP have more settings than the kernel models. Ties keep the first grid entry.

### Cell 2: Run selection, refit and evaluation

Fit every model on the same splits and inputs, pick parameters by validation balanced accuracy, refit on development rows and score the evaluation rows. This takes a few minutes, mostly for the exact kernels.

In [ ]:
metrics, predictions = ec.run(ROOT, PROJECTED)
meta = pd.read_json(ROOT / 'results/expanded/metadata.json', typ='series')
print('Stage 4 models reproduced:', meta['stage4_reproduced'])
display(metrics[['task','dimension','model','family','params','validation_balanced_accuracy','balanced_accuracy','macro_f1']])

## Statistical tests
Every model is compared with the existing ZZ-map QSVM on the same 500 evaluation rows.

- **Paired date-block bootstrap** (2,000 resamples): whole evaluation dates are resampled, so the five stocks on one date stay together. It gives a 95% interval for each model's balanced accuracy and for its difference from the QSVM, plus a two-sided p-value. It does not model dependence between dates.
- **Exact McNemar test**: compares which rows each model gets right. It treats rows as independent, so it is a secondary check only.
- **Holm correction**: applied to the bootstrap p-values within each task and dimension.

With 100 evaluation dates, intervals of several percentage points are expected. A non-significant difference is not evidence that two models are equal.

### Cell 3: Compute intervals and tests

Run the bootstrap and McNemar tests and build one comparison table per task and dimension.

In [ ]:
table = ec.summarise(ROOT, metrics, ec.significance(predictions))
cols = ['model','family','balanced_accuracy','ba_ci_low','ba_ci_high','diff_vs_qsvm_pp','diff_ci_low_pp','diff_ci_high_pp','p_bootstrap','p_holm','p_mcnemar']
for (task, d), g in table.groupby(['task','dimension']):
    print(f'\n{task}, {d} components (chance = {g.chance.iloc[0]:.3f})')
    display(g[cols].round(4).reset_index(drop=True))

### Cell 4: Plot intervals

Show each model's balanced accuracy with its 95% interval, and its difference from the ZZ-map QSVM.

In [ ]:
ec.figures(ROOT, table)

### Cell 5: Check seed sensitivity

Refit the random forest and MLP with five other seeds at their selected parameters and report the spread in evaluation balanced accuracy.

In [ ]:
seeds = pd.read_csv(ROOT / 'results/expanded/seed_check.csv')
display(seeds.groupby(['task','dimension','model']).balanced_accuracy.agg(['mean','std','min','max']).round(4))

### Cell 6: Summarise what the tests support

For each task and dimension, list the best model by evaluation balanced accuracy, whether its interval excludes chance, and how many comparisons with the QSVM survive the Holm correction.

In [ ]:
rows = []
for (task, d), g in table.groupby(['task','dimension']):
    best = g.sort_values('balanced_accuracy', ascending=False).iloc[0]
    q = g[g.model == ec.REFERENCE].iloc[0]
    others = g[g.model != ec.REFERENCE]
    rows.append(dict(task=task, dimension=d, best_model=best.model, best_ba=round(best.balanced_accuracy, 4),
        best_ci_excludes_chance=bool(best.ba_ci_low > best.chance), qsvm_ba=round(q.balanced_accuracy, 4),
        qsvm_ci_excludes_chance=bool(q.ba_ci_low > q.chance), qsvm_rank=int((g.balanced_accuracy > q.balanced_accuracy).sum() + 1),
        models=len(g), holm_significant_vs_qsvm=int((others.p_holm < 0.05).sum())))
summary = pd.DataFrame(rows); summary.to_csv(ROOT / 'reports/expanded/summary.csv', index=False)
display(summary)

## Interpretation guardrails
- Read the results within each task and dimension. Binary and three-class scores have different chance levels (0.50 and 0.33) and are not equal-difficulty tasks.
- A model that ranks above the QSVM without a Holm-significant difference has not been shown to be better.
- The Z-map versus ZZ-map pair is the only controlled quantum ablation here: it changes the entangling layer and nothing else.
- All quantum results come from exact, noiseless simulation on 2 or 4 qubits. Nothing here shows quantum advantage, and nothing measures trading profitability.